# Download PropertyLens artifacts from Hugging Face

Companion to `upload_artifacts_to_hf.ipynb`. Pulls every large artifact the backend needs at runtime into the exact folder layout that `backend/main.py` and friends look for.

| HF repo | Pulls | Lands at (local) | Why |
|---|---|---|---|
| `PropertyLens/final-propertylens-models` | `artifacts/**` | `data/artifacts/**` | `app_state.ARTIFACTS_ROOT` |
| `PropertyLens/final-propertylens-models` | `05_photo_layer/artifacts/**` | `hf_data/05_photo_layer/artifacts/**` | First search dir in `yc_photo_condition.load_condition_model()` |
| `PropertyLens/final-dataset` | `feature_data/**` | `data/feature_data/**` | `app_state.FEATURE_LAYER_OUTPUTS` |
| `PropertyLens/final-dataset` | `amenities/**` | `data/amenities/**` | Raw schooling extract used by preprocessing |

After this notebook completes, run the backend (`uvicorn main:app --reload --port 8000` from `backend/`) and every artifact load will succeed without further setup.

**Prereqs**
- `HF_TOKEN` in `.env` (read scope is enough — write only needed for the upload notebook)
- `huggingface_hub` installed (`pip install -q huggingface_hub python-dotenv`)

## 1 — Setup

In [ ]:
%pip install -q huggingface_hub python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import HfApi, snapshot_download

REPO_ROOT = Path.cwd().resolve()
load_dotenv(REPO_ROOT / ".env")

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — proceeding anonymously (only works if both repos are public).")

api = HfApi(token=HF_TOKEN)
if HF_TOKEN:
    whoami = api.whoami()
    print(f"Logged in as: {whoami.get('name')} ({whoami.get('email', 'no-email')})")
print(f"Repo root: {REPO_ROOT}")

## 2 — Repo + path config

Each entry maps a **HF repo path** → **local destination** so files land where backend code expects them.

In [ ]:
MODEL_REPO   = "PropertyLens/final-propertylens-models"   # type: model
DATASET_REPO = "PropertyLens/final-dataset"               # type: dataset

DOWNLOADS = [
    {
        "label": "Hybrid ensemble + XAI artifacts",
        "repo_id": MODEL_REPO,
        "repo_type": "model",
        "local_dir": REPO_ROOT / "data",
        "allow_patterns": ["artifacts/**"],
    },
    {
        "label": "Photo condition model (.pth + meta)",
        "repo_id": MODEL_REPO,
        "repo_type": "model",
        "local_dir": REPO_ROOT / "hf_data",
        "allow_patterns": ["05_photo_layer/artifacts/**"],
    },
    {
        "label": "Feature tables (train / test / full)",
        "repo_id": DATASET_REPO,
        "repo_type": "dataset",
        "local_dir": REPO_ROOT / "data",
        "allow_patterns": ["feature_data/**"],
    },
    {
        "label": "Raw schooling extract",
        "repo_id": DATASET_REPO,
        "repo_type": "dataset",
        "local_dir": REPO_ROOT / "data",
        "allow_patterns": ["amenities/**"],
    },
]

for d in DOWNLOADS:
    d["local_dir"].mkdir(parents=True, exist_ok=True)
    print(f"  {d['label']}\n    {d['repo_id']} ({d['repo_type']}) {d['allow_patterns']}\n    → {d['local_dir']}")

## 3 — Download

`snapshot_download` is incremental — re-running this cell is safe and skips files that are already up to date.

In [ ]:
for d in DOWNLOADS:
    print(f"⬇️  {d['label']}")
    print(f"   {d['repo_id']} ({d['repo_type']}) — patterns: {d['allow_patterns']}")
    path = snapshot_download(
        repo_id=d["repo_id"],
        repo_type=d["repo_type"],
        token=HF_TOKEN,
        local_dir=str(d["local_dir"]),
        allow_patterns=d["allow_patterns"],
    )
    print(f"   ✓ done → {path}\n")

print("All downloads complete.")

## 4 — Verify (backend can find every artifact)

Checks the exact files `backend/main.py` loads at startup. Anything marked `MISSING` will break the backend.

In [ ]:
REQUIRED = [
    # Hybrid ensemble + meta (loaded by hybrid_inference + main)
    "data/artifacts/hybrid_cluster_bundle.joblib",
    "data/artifacts/hybrid_cluster_feature_columns.json",
    "data/artifacts/hybrid_cluster_meta.json",
    # XAI core (loaded by main.py at startup)
    "data/artifacts/hybrid_xai/cbr_index.joblib",
    "data/artifacts/hybrid_xai/cbr_scaler.joblib",
    "data/artifacts/hybrid_xai/cbr_training_data.parquet",
    "data/artifacts/hybrid_xai/cbr_features.json",
    "data/artifacts/hybrid_xai/global_shap_cache.json",
    "data/artifacts/hybrid_xai/rules.json",
    "data/artifacts/hybrid_xai/lime_training_data.joblib",
]

OPTIONAL = [
    # Newer XAI artifacts (graceful skip if missing)
    "data/artifacts/hybrid_xai/global_shap_by_cluster.json",
    "data/artifacts/hybrid_xai/composite_treeshap_global_importance.json",
    "data/artifacts/hybrid_xai/cluster_profiles.json",
    "data/artifacts/hybrid_xai/shap_explainers.joblib",
    "data/artifacts/hybrid_xai/surrogate_model.joblib",
]

DATA_DIRS = [
    # Backend feature-table glob; needs at least one hdb_feature_table_*.csv
    ("data/feature_data/02_feature_layer/training/outputs", "hdb_feature_table_*.csv"),
]

PHOTO_DIRS = [
    # yc_photo_condition.load_condition_model() searches hf_data first
    ("hf_data/05_photo_layer/artifacts", "condition_model_*.pth"),
]

ok = True

print("── Required ──")
for rel in REQUIRED:
    p = REPO_ROOT / rel
    mark = "✅" if p.exists() else "❌ MISSING"
    size = f"{p.stat().st_size / 1e6:,.2f} MB" if p.exists() else ""
    print(f"  {mark}  {rel}  {size}")
    if not p.exists():
        ok = False

print("\n── Optional (newer / nice-to-have) ──")
for rel in OPTIONAL:
    p = REPO_ROOT / rel
    mark = "✅" if p.exists() else "⏭️  not present"
    size = f"{p.stat().st_size / 1e6:,.2f} MB" if p.exists() else ""
    print(f"  {mark}  {rel}  {size}")

print("\n── Globbed dirs (need at least one match) ──")
for rel_dir, pattern in DATA_DIRS + PHOTO_DIRS:
    d = REPO_ROOT / rel_dir
    matches = sorted(d.glob(pattern)) if d.exists() else []
    if matches:
        print(f"  ✅  {rel_dir}/{pattern} → {len(matches)} match(es)")
        for m in matches[:5]:
            size = f"{m.stat().st_size / 1e6:,.2f} MB"
            print(f"      {m.name}  ({size})")
        if len(matches) > 5:
            print(f"      … (+{len(matches) - 5} more)")
    else:
        print(f"  ❌ MISSING  {rel_dir}/{pattern}")
        ok = False

print("\n" + "=" * 60)
print("✅  Backend has everything it needs to start." if ok else "❌  Some required artifacts are missing — backend will fail at startup.")